In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3 وحزمة Qiskit SDK

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
```
</details>

توفر حزمة Qiskit SDK بعض الأدوات للتحويل بين تمثيلات OpenQASM للبرامج الكمية وفئة QuantumCircuit. لاحظ أن هذه الأدوات لا تزال في مرحلة استكشافية من التطوير وستستمر في التطور مع زيادة دعم Qiskit لقدرات الدوائر الديناميكية التي يعبّر عنها OpenQASM 3.

> **Note:** هذه الوظيفة لا تزال في المرحلة الاستكشافية. لذلك، من المرجح أن تتطور الصياغة والقدرات.

## استيراد برنامج OpenQASM 3 إلى Qiskit
يجب عليك تثبيت الحزمة `qiskit_qasm3_import` لاستخدام هذه الوظيفة. قم بالتثبيت باستخدام الأمر التالي.

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

حاليًا، تتوفر دالتان رفيعتا المستوى للاستيراد من OpenQASM 3 إلى Qiskit. هاتان الدالتان هما `load()` التي تأخذ اسم ملف، و`loads()` التي تأخذ البرنامج نفسه كسلسلة نصية:

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

في هذا المثال، نعرّف برنامجًا كميًا باستخدام OpenQASM 3، ونستخدم `loads()` لتحويله مباشرةً إلى QuantumCircuit:

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![Output of the previous code cell](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## التصدير إلى OpenQASM 3
يمكنك تصدير كود Qiskit إلى OpenQASM 3 باستخدام `dumps()` للتصدير إلى سلسلة نصية، أو `dump()` للتصدير إلى ملف.

### مثال باستخدام `dumps()`